# RMSNorm
*Root Mean Square Layer Normalization — the efficient normalization in LLaMA, Mistral, and Gemma.*

**Companion kernels:** `v0_naive.py` → `sm89_v1_block_per_row.py` → `sm89_v2_smem_tree_reduce.py`


## What Is RMSNorm?

**In plain English:** RMSNorm rescales each token's vector so its "energy" (measured by the root-mean-square) equals 1, then multiplies by a learned per-dimension scale factor.

Think of it as **automatic volume control**: whatever the signal level coming in, normalize it to a consistent energy, then apply a learned gain.

**Why it exists:** Raw activations in deep networks tend to grow or shrink exponentially across layers — training becomes unstable. Normalization pins the magnitude so every layer receives well-scaled input.

**Where it appears:** Before every attention and FFN sub-layer in LLaMA, LLaMA 2/3, Mistral, Gemma, Qwen, GLM-4.


In [ ]:
import math
print('ready')

## Worked Example: Step by Step

**Input vector** (one token's representation):
$$x = [3.0,\ 4.0,\ 0.0]$$

**Learned scale:**
$$\gamma = [1.0,\ 1.0,\ 1.0] \quad \text{(identity — no scaling yet)}$$

| Step | Operation | Result |
|------|-----------|--------|
| 1 | $x^2 = [3^2,\ 4^2,\ 0^2]$ | $[9.0,\ 16.0,\ 0.0]$ |
| 2 | mean of $x^2 = (9+16+0)/3$ | $8.333$ |
| 3 | $\text{RMS} = \sqrt{8.333 + \varepsilon}$ | $2.887$ |
| 4 | $\text{inv\_rms} = 1 / 2.887$ | $0.346$ |
| 5 | output $= x \cdot \text{inv\_rms} \cdot \gamma$ | $[1.039,\ 1.386,\ 0.000]$ |

**Verification:** $\text{RMS}(\text{output}) = \sqrt{(1.039^2 + 1.386^2 + 0^2)/3} = \sqrt{1.0} = 1.0$ ✅


In [ ]:
x = [3.0, 4.0, 0.0]
gamma = [1.0, 1.0, 1.0]
eps = 1e-5
d = len(x)

sq = [xi**2 for xi in x]
mean_sq = sum(sq) / d
rms = math.sqrt(mean_sq + eps)
inv_rms = 1.0 / rms
output = [xi * inv_rms * gi for xi, gi in zip(x, gamma)]

print(f"x²         = {sq}")
print(f"mean(x²)   = {mean_sq:.4f}")
print(f"RMS        = {rms:.4f}")
print(f"inv_rms    = {inv_rms:.4f}")
print(f"output     = {[round(v, 4) for v in output]}")

# Verify: RMS of output should be ≈ 1.0
rms_out = math.sqrt(sum(v**2 for v in output) / d)
print(f"\nRMS(output) = {rms_out:.6f}  (should be 1.0 ✓)")
assert abs(rms_out - 1.0) < 1e-5


## The Formula

$$\boxed{\text{RMSNorm}(x)_i = \frac{x_i}{\text{RMS}(x)} \cdot \gamma_i}$$

where

$$\text{RMS}(x) = \sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \varepsilon}$$

### Symbol Definitions

| Symbol | Name | Value in our example |
|--------|------|----------------------|
| $x_i$ | input activation at index $i$ | $[3.0, 4.0, 0.0]$ |
| $d$ | dimension of the vector | $3$ |
| $\sum_{i=1}^{d} x_i^2$ | sum of squares | $9 + 16 + 0 = 25$ |
| $\frac{1}{d}(\cdots)$ | mean of squares | $25/3 = 8.333$ |
| $\sqrt{\cdots}$ | square root | $2.887$ |
| $\varepsilon$ | small constant (epsilon) | $10^{-5}$, prevents division by zero |
| $\gamma_i$ | learned scale parameter | Initialized to 1, learned during training |

### 🗣️ Say It Out Loud
> *"RMSNorm of x at index i equals x-sub-i divided by the RMS of x, times gamma-sub-i. The RMS is the square root of the mean of the squares, plus a tiny epsilon for safety."*

**Tutor's gloss:** "We're measuring the 'energy' of the vector with $\frac{1}{d}\sum x_i^2$ — that's just the average of the squared values. Taking the square root gives us the root-mean-square. Dividing each element by this normalizes the total energy to 1. Then $\gamma$ lets the model re-scale however it learned to prefer."


## RMSNorm vs LayerNorm

LayerNorm (Ba et al., 2016) normalizes each vector to unit variance *and* zero mean, then applies a learned scale and shift:

$$\text{LN}(x)_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \varepsilon}} \cdot \gamma_i + \beta_i$$

where $\mu = \frac{1}{d}\sum x_i$ and $\sigma^2 = \frac{1}{d}\sum (x_i - \mu)^2$.

**RMSNorm drops both the mean subtraction and the $\beta$ shift parameter:**

$$\text{RMSNorm}(x)_i = \frac{x_i}{\text{RMS}(x)} \cdot \gamma_i$$

### Why can you drop both, and how is quality maintained?

**Dropping the mean subtraction ($\mu$ term):**

Subtracting the mean "re-centers" the output distribution — it shifts values so their average is zero. In a transformer, though, the projection layers (Q, K, V, O, gate, up, down) each have their own bias parameters that already handle shifting. Centering the distribution inside the norm is redundant: you shift it here, and the next layer shifts it back to where it needs to be. Removing this operation means one fewer full scan over the data.

**Dropping $\beta$ (the additive shift):**

$\gamma$ rescales — it multiplies each output dimension by a learned constant. $\beta$ shifts — it adds a learned constant. They seem like a natural pair, and in early architectures they were both used. But the same redundancy argument applies: the next layer's bias does the shifting. Zhang & Sennrich (2019) tested removing $\beta$ on translation benchmarks and found **no quality loss**. Modern LLMs (LLaMA, Mistral, Qwen) all dropped $\beta$ entirely.

### The concrete cost saving

| What LayerNorm does | What RMSNorm does |
|---|---|
| Compute mean $\mu$ (one pass) | — |
| Compute variance $\sigma^2$ relative to $\mu$ (one more pass) | — |
| Compute RMS (= $\sigma$ if mean=0) | Compute RMS (one pass) |
| Apply $\gamma$ scale + $\beta$ shift | Apply $\gamma$ scale only |

LayerNorm needs to compute and subtract the mean before it can compute the variance. That's two dependent passes. RMSNorm skips the mean entirely — just one pass for the sum of squares. **Result: 7–64% faster** depending on hidden dimension, with no measurable quality difference on transformer architectures.

### Same vs Different Example

In [ ]:
def rmsnorm(x, gamma, eps=1e-5):
    d = len(x)
    rms = math.sqrt(sum(xi**2 for xi in x) / d + eps)
    return [xi / rms * gi for xi, gi in zip(x, gamma)]

def layernorm(x, gamma, beta, eps=1e-5):
    d = len(x)
    mu = sum(x) / d
    var = sum((xi - mu)**2 for xi in x) / d
    std = math.sqrt(var + eps)
    return [(xi - mu) / std * gi + bi for xi, gi, bi in zip(x, gamma, beta)]

gamma = [1.0] * 3
beta  = [0.0] * 3

# Zero-mean input: LN and RMSNorm give same result
x_zero_mean = [1.0, -1.0, 0.0]
rms_out = rmsnorm(x_zero_mean, gamma)
ln_out  = layernorm(x_zero_mean, gamma, beta)
print(f"x = {x_zero_mean}  (zero mean)")
print(f"  RMSNorm: {[round(v,4) for v in rms_out]}")
print(f"  LayerNorm: {[round(v,4) for v in ln_out]}")
print(f"  Same? {all(abs(a-b)<1e-5 for a,b in zip(rms_out, ln_out))}\n")

# Non-zero-mean input: they DIFFER
x_shifted = [3.0, 4.0, 0.0]
rms_out2 = rmsnorm(x_shifted, gamma)
ln_out2  = layernorm(x_shifted, gamma, beta)
print(f"x = {x_shifted}  (mean = {sum(x_shifted)/3:.2f}, non-zero)")
print(f"  RMSNorm: {[round(v,4) for v in rms_out2]}")
print(f"  LayerNorm: {[round(v,4) for v in ln_out2]}")
print(f"  Same? {all(abs(a-b)<1e-5 for a,b in zip(rms_out2, ln_out2))}")
print("  → They differ because LN subtracts the mean first")


## Optimization Ladder

| Version | Threads | Passes | Key idea |
|---------|---------|--------|----------|
| `v0_naive` | 1 | 2 | One thread: sum-of-squares, then normalize |
| `sm89_v1_block_per_row` | 128 | 2 | Parallel partial sums, tree reduce in SMEM |
| `sm89_v2_smem_tree_reduce` | 128 | 2 | Vectorized loads (`float4`), better SMEM layout |

**Why 2 passes?**
RMSNorm needs the full RMS before it can normalize any element.
Pass 1: compute $\sum x_i^2$ → get RMS.
Pass 2: apply $x_i / \text{RMS} \cdot \gamma_i$.
(Online single-pass isn't possible here unlike softmax.)


## RTX 4080: RMSNorm is Strongly Memory-Bound

RMSNorm does about 5 FLOPs per element: square, accumulate, sqrt, reciprocal, multiply.
It reads and writes 4 bytes per element in BF16 (2 bytes in, 2 bytes out), plus the tiny weight vector.
That works out to roughly **0.8 FLOP/byte**.

The RTX 4080 sits at a "ridge point" of ~151 FLOP/byte — the boundary between memory-limited and compute-limited.
We're 188× below that ridge. **The GPU is waiting on memory, not math.**

What this means for kernel design:
- Saving FLOPs is pointless — compute is not the bottleneck
- The only real wins come from reading/writing memory faster or fewer times
- `float4` vectorized loads (v2 kernel) pack 4 BF16 values into one load instruction → directly hits the bandwidth ceiling

| Kernel | BW utilization | Bottleneck removed |
|--------|---------------|-------------------|
| v0_naive (1 thread) | ~5% | Thread count — one thread can't saturate memory |
| sm89_v1 (128 threads) | ~40–60% | Thread count — now many threads stream in parallel |
| sm89_v2 (vectorized) | ~70–85% | Load granularity — 4 values per instruction |

## CuTeDSL Kernel Walkthrough

Here is the block-per-row kernel (`sm89_v1`) with every line explained in plain English:

```python
@cute.kernel
def rmsnorm_kernel(mX, mW, mOut, eps):
    row = blockIdx.x    # "which token am I?" — one GPU block handles one row
    tid = threadIdx.x   # "which thread am I within this block?" — 0 to 127
    D   = mX.shape[1]   # hidden dimension (4096 for Qwen3-8B)

    # --- Pass 1: compute partial sum-of-squares ---
    # Thread 0 handles elements 0, 128, 256, ...
    # Thread 1 handles elements 1, 129, 257, ...  etc.
    # Every element gets covered, work is split evenly.
    ss = 0.0
    for i in cutlass.range(tid, D, 128):
        xi = mX[row, i]   # read one BF16 value from global memory
        ss += xi * xi     # accumulate in FP32 (BF16 upcasts automatically)

    # --- Tree reduce: 128 partial sums → 1 global sum ---
    # Think of it like a tournament bracket:
    # Round 1: threads 0-63 each add in thread 64-127's value  (64 additions)
    # Round 2: threads 0-31 each add in thread 32-63's value   (32 additions)
    # ...7 rounds total, then smem[0] holds the sum for the whole row
    smem[tid] = ss
    cute.sync_threads()   # wait: everyone must finish writing before anyone reads
    for half in [64, 32, 16, 8, 4, 2, 1]:
        if tid < half:
            smem[tid] += smem[tid + half]
        cute.sync_threads()

    inv_rms = cute.rsqrt(smem[0] / D + eps)
    # rsqrt = "reciprocal square root" — the GPU has a dedicated 1-cycle instruction for this

    # --- Pass 2: normalize and write output ---
    for i in cutlass.range(tid, D, 128):
        mOut[row, i] = mX[row, i] * inv_rms * mW[i]
        # Note: this re-reads mX from global memory. The v2 kernel caches it in registers.
```

**The v2 upgrade (sm89_v2_smem_tree_reduce):**
Replace the scalar load `mX[row, i]` with a vectorized load that grabs 4 BF16 values at once:
```python
# Instead of:  xi = mX[row, i]            # 1 value, 2 bytes, 1 instruction
# Do this:     xi_vec = mX.load_v4(row, i) # 4 values, 8 bytes, 1 instruction
```
Same number of instructions, 4× the data throughput. This is the main difference between v1 and v2.

## Two Shapes in Qwen3-8B: Layer Norm vs QK-Norm

Qwen3-8B runs RMSNorm in two places, and each needs a different kernel strategy.

**Layer norm** — applied to the hidden state before every attention and FFN block:
- Shape: `[seq_len, 4096]`  — one row per token, 4096 elements wide
- At decode (seq_len=1): a single row of 4096 elements
- At prefill (seq_len=2048): 2048 rows processed in parallel

**QK-norm** — Qwen3 normalizes Q and K per head before the attention dot-product:
- Q shape: `[seq_len × 32_heads, 128]`  — 32× more rows, each only 128 wide
- K shape: `[seq_len × 8_heads,  128]`  — same width, fewer rows

**Why D=128 changes everything:**
One warp = 32 threads. At D=128, each thread owns 4 elements (128 ÷ 32 = 4).
The entire row fits inside a single warp. The reduction never needs shared memory.

| Use case   | D    | Elements/thread | Reduction           | SMEM needed |
|------------|------|-----------------|---------------------|-------------|
| Layer norm | 4096 | 32              | Inter-warp via SMEM | 128–512 B   |
| QK-norm    | 128  | 4               | Intra-warp only     | **0 bytes** |

QK-norm is an entirely different kernel class — zero shared memory, zero sync barriers.

## PTX: Warp Shuffle Reduction

### Two ways for GPU threads to share a number

**The chalkboard (shared memory):** Imagine 128 people in a room, each holding a number, and a shared whiteboard on the wall. To sum all 128 numbers, one approach is to have everyone write their value on the board. But you can't read until everyone has written — so you call for silence (a sync barrier). Then active participants each read their neighbor's value and write a new partial sum. Seven rounds of write-sync-read collapse 128 values to 1.

Each round costs: the write (~4 cycles), the barrier drain (~20 cycles waiting for all 128 threads), and the read (~20 cycles latency). About 40 cycles per round × 7 rounds = ~280 cycles total.

**The whisper (warp shuffle):** A warp is 32 threads that run in true hardware lockstep on one SM unit — they share a physical register file. Instead of writing to a board and waiting, thread 5 can just *read thread 21's register directly*, like whispering across a table. No memory involved at all. No barriers. Just a register-to-register transfer in ~4 cycles.

The PTX instruction is `shfl.sync.bfly.b32` (butterfly = XOR-based pairing).
In CUDA C: `__shfl_xor_sync`. Five rounds collapse 32 values to 1:

```
Round 1 (offset 16): thread 0 += thread 16,  thread 1 += thread 17,  ...
Round 2 (offset  8): thread 0 += thread 8,   thread 1 += thread 9,   ...
Round 3 (offset  4): thread 0 += thread 4,   ...
Round 4 (offset  2): thread 0 += thread 2,   ...
Round 5 (offset  1): thread 0 += thread 1    ← thread 0 now holds the total
```

No `sync_threads()`. No shared memory write. No shared memory read.
Five register-to-register passes — the fastest possible reduce.

**Comparing the two approaches:**

| | SMEM tree reduce | Warp shuffle |
|---|---|---|
| Rounds needed | 7 (for 128 threads) | 5 (for 32 threads) |
| Cost per round | ~40 cycles (write + barrier + read) | ~4 cycles (register transfer) |
| SMEM writes | 128 | 0 |
| Sync barriers | 7 | 0 |
| Estimated total latency | ~280 cycles | ~20 cycles |
| Works across warps? | ✅ | ❌ (one warp only) |

---

**QK-norm (D=128): warp-only kernel**

At D=128 with 32 threads (one warp), each thread owns 4 elements (128 ÷ 32 = 4). The entire row fits inside a single warp — no chalkboard needed, only whispers:

```python
@cute.kernel
def qk_rmsnorm_kernel(mX, mW, mOut, eps):
    row = blockIdx.x   # one warp per row (blockDim.x = 32)
    k   = threadIdx.x  # 0..31, each thread owns 4 elements
    D   = 128

    # Each thread accumulates sum-of-squares for its 4 elements
    ss = 0.0
    for i in [k, k+32, k+64, k+96]:
        xi = mX[row, i]
        ss += xi * xi

    # Five shuffle rounds — zero SMEM, zero sync barriers
    ss += cute.shfl_xor(ss, 16)
    ss += cute.shfl_xor(ss, 8)
    ss += cute.shfl_xor(ss, 4)
    ss += cute.shfl_xor(ss, 2)
    ss += cute.shfl_xor(ss, 1)
    # Thread 0 has the total. Broadcast it back to every thread in the warp.
    inv_rms = cute.rsqrt(cute.shfl_broadcast(ss, 0) / D + eps)

    for i in [k, k+32, k+64, k+96]:
        mOut[row, i] = mX[row, i] * inv_rms * mW[i]
```

**Layer norm (D=4096): hybrid strategy**

At D=4096 with 128 threads, there are 4 warps. Each warp first whispers internally (5-round shuffle) to produce one partial sum per warp. Then only those 4 partial sums go onto the chalkboard for the inter-warp step:

```python
# After intra-warp shuffle, thread 0 of each warp holds its warp's partial sum
if lane_id == 0:
    smem[warp_id] = partial_sum   # 4 writes total (one per warp)
cute.sync_threads()               # 1 barrier — only need to sync these 4 writes

# Only the first warp finishes the job — a whisper round over the 4 partial sums
if warp_id == 0:
    val = smem[lane_id] if lane_id < 4 else 0.0
    val += cute.shfl_xor(val, 2)
    val += cute.shfl_xor(val, 1)
    smem[0] = cute.rsqrt(val / D + eps)
cute.sync_threads()               # 1 barrier — broadcast inv_rms

inv_rms = smem[0]   # every thread reads the same value
```

**The win:** the pure SMEM tree reduce uses 128 writes and 7 barriers.
This hybrid uses **4 writes and 2 barriers** — and the intra-warp whispers cost ~20 cycles vs ~280 for a full chalkboard tree.

## Where RMSNorm Lives: The Full Qwen3-8B Decode Step

When a user sends a query, the text is tokenized first (outside the model).
Each word or subword maps to an integer token ID from a vocabulary of 151,936 entries.
From that point on, everything below is what the model does — **every single token generated**.

---

### Step 0 — Embedding lookup
```
input_ids: [token_id]                  shape: [1, 1]   (batch=1, new token=1)
x = embed_tokens(input_ids)            shape: [1, 1, 4096]
```
The embedding table is `[151936, 4096]`. Each token ID picks one row.
This is a lookup, not a multiply. Output `x` is the starting hidden state.

### What does `x` mean here?

`x` is the hidden representation of the current token after embedding.
It is a vector of 4096 numbers.

You can think of it as the token's current “meaning” in the model's internal space.
At the beginning of the forward pass, it is just the embedding vector.
After each layer, it is updated and becomes a richer representation.

So the line

```text
x: [1, 1, 4096]
```

means:
- one example in the batch
- one token position in the sequence
- 4096 features for that token

This is the same idea as saying: “one token has a 4096-dimensional feature vector.”

---

### Steps 1–36 — Transformer layers (repeated 36 times)

Each of the 36 layers runs the same pattern:

```
┌─────────────────────────────────────────────────────────────────┐
│  x: [1, 1, 4096]  ← hidden state coming in                     │
│                                                                 │
│  ① RMSNorm (ln1)          weight [4096]     → normed [1,1,4096]│
│  ② Q projection            weight [4096,4096] → q [1,1,32,128] │
│  ③ K projection            weight [1024,4096] → k [1,1, 8,128] │
│  ④ V projection            weight [1024,4096] → v [1,1, 8,128] │
│  ⑤ RMSNorm (q_norm)       weight [128]      → q normalized     │
│  ⑥ RMSNorm (k_norm)       weight [128]      → k normalized     │
│  ⑦ RoPE                   (no weights)      → q,k rotated      │
│  ⑧ Append to KV cache     k,v stored for future decode steps   │
│  ⑨ GQA attention          (no weights)      → [1,1,4096]       │
│  ⑩ O projection            weight [4096,4096] → [1,1,4096]     │
│  ⑪ Residual add            x = x + attn_out  → [1,1,4096]      │
│                                                                 │
│  ⑫ RMSNorm (ln2)          weight [4096]     → normed [1,1,4096]│
│  ⑬ Gate projection         weight [12288,4096] → [1,1,12288]   │
│  ⑭ Up projection           weight [12288,4096] → [1,1,12288]   │
│  ⑮ SwiGLU activation      (no weights)      → [1,1,12288]      │
│  ⑯ Down projection         weight [4096,12288] → [1,1,4096]    │
│  ⑰ Residual add            x = x + ffn_out  → [1,1,4096]       │
└─────────────────────────────────────────────────────────────────┘
```

### What those projection lines mean

The three lines below are the main attention projection steps:

```text
② Q projection  weight [4096, 4096] → q [1,1,32,128]
③ K projection  weight [1024, 4096] → k [1,1, 8,128]
④ V projection  weight [1024, 4096] → v [1,1, 8,128]
```

They take the current hidden state `x` and turn it into the query, key, and value representations used for attention.

In plain English:
- the current hidden vector of size 4096 is transformed into
- a set of query vectors for the 32 attention heads
- a set of key vectors for the 8 KV heads
- a set of value vectors for the 8 KV heads

The shapes are telling you how many heads and how many values each head carries:
- `q` has 32 heads, each of width 128
- `k` and `v` each have 8 heads, each of width 128

So the model is not just “changing the vector”; it is splitting it into separate head-wise representations that attention will use.

---

### Step 37 — Final RMSNorm
```
x = self.norm(x)      weight [4096]   → [1, 1, 4096]
```

### Step 38 — LM Head
```
logits = lm_head(x)   weight [151936, 4096]  →  [1, 1, 151936]
```
This IS a matrix multiply: one 4096-wide vector dotted with 151,936 weight rows.
Result: a score for every word in the vocabulary.

### Step 39 — Sampling
```
next_token_id = sample(logits[:, -1, :])   → scalar integer
```
Argmax (greedy) or top-p sampling picks the next token ID.
That ID feeds back into Step 0 for the next decode step.

---

### RMSNorm call count per decode step

| Location | Weight shape | Calls per layer | Total (36 layers) |
|----------|-------------|-----------------|-------------------|
| ln1 (pre-attention) | [4096] | 1 | 36 |
| q_norm (QK-norm Q) | [128] | 1 | 36 |
| k_norm (QK-norm K) | [128] | 1 | 36 |
| ln2 (pre-FFN) | [4096] | 1 | 36 |
| final norm | [4096] | — | 1 |
| **Total** | | | **145 calls** |

The 145 calls happen every single token generated.
They are tiny individually, but at long context with many tokens, they add up.


## The Weight γ: A 1D Scale Vector, Not a Matrix

This is the actual `RMSNorm` implementation in this project (`baseline/norm.py`):

```python
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        self.weight = nn.Parameter(torch.ones(dim))   # ← 1D vector, shape [dim]

    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * norm * self.weight                 # ← element-wise, not matmul
```

### What does `x` mean here?

`x` is the hidden-state vector for one token at one point inside the model.
It is the representation of that token after the previous layer has produced it.

In this notebook, we mostly use a tiny toy example:

```python
x = [3.0, 4.0, 0.0]
```

That means: one token has a 3-dimensional activation vector.
In the real Qwen3 model, that vector is much larger:

- `x` has shape `[1, 1, 4096]` during decode
- that means:
  - batch size = 1
  - sequence length = 1
  - hidden size = 4096

So the model is processing one token at a time, and that token has 4096 features.

### Why the shape is `[1, 1, 4096]`

Think of it like this:

- the first `1` = one example in the batch
- the second `1` = one token position in the sequence
- the last `4096` = the hidden representation of that token

For a prefill step with many tokens, the shape could be larger, for example `[1, 2048, 4096]`.

### Why the multiplication is element-wise

`self.weight` is a 1D vector of shape `[4096]`.
It contains one learned scale value per hidden dimension.

So for each dimension `i`:

$$
\text{output}_i = x_i \cdot \frac{1}{\mathrm{RMS}(x)} \cdot \gamma_i
$$

where:
- `x_i` = the $i$-th value of the hidden vector
- `1 / RMS(x)` = one single scalar computed from the whole vector
- `gamma_i` = the learned scale for that dimension

This is why the operation is called an element-wise or Hadamard product:

- it is not a dot product
- it is not a matrix multiply
- it is just one scalar scaling factor applied to every element, plus a per-dimension scale vector

### A very concrete example

If the hidden vector is:

```python
x = [3.0, 4.0, 0.0]
```

and the learned scale is:

```python
gamma = [1.0, 1.0, 1.0]
```

then each output value is computed independently as:

```python
output[i] = x[i] * inv_rms * gamma[i]
```

No matrix multiplication is involved.

---

### Where the weight comes from

The weight is initialized to `torch.ones(dim)` — all 1.0.
At that initialization, RMSNorm is a pure normalization with no scaling effect.
During training, `weight` is updated by the optimizer to learn per-dimension gain values.
At inference, it's loaded from the model checkpoint:

```python
# In the checkpoint, the weight lives at paths like:
model.layers[0].input_layernorm.weight    # ln1, shape [4096]
model.layers[0].post_attention_layernorm.weight   # ln2, shape [4096]
model.layers[0].self_attn.q_norm.weight   # q_norm, shape [128]
model.layers[0].self_attn.k_norm.weight   # k_norm, shape [128]
model.norm.weight                         # final norm, shape [4096]
```

---

### What IS a matrix multiplication here?

The actual matrix multiplications are the **projection weights** in the attention block:

| Weight | Shape | Operation |
|--------|-------|-----------|
| `q_proj.weight` | [4096, 4096] | `x [1,4096] × W [4096,4096] → q [1,4096]` |
| `k_proj.weight` | [1024, 4096] | `x [1,4096] × W [4096,1024] → k [1,1024]` |
| `v_proj.weight` | [1024, 4096] | `x [1,4096] × W [4096,1024] → v [1,1024]` |
| `o_proj.weight` | [4096, 4096] | `attn_out [1,4096] × W → [1,4096]` |
| `lm_head.weight` | [151936, 4096] | `x [1,4096] × W [4096,151936] → logits [1,151936]` |

These are the GEMMs. RMSNorm is not one of them.

---

### Memory footprint of RMSNorm weights (all 145 of them)

```
72 × [4096] in BF16  = 72 × 4096 × 2 = 589 KB   (ln1, ln2, final)
72 × [128]  in BF16  = 72 × 128  × 2 =  18 KB   (q_norm, k_norm)
Total RMSNorm weights:  ≈ 0.6 MB
```

Compare to the MLP weights alone: 3 matrices × 4096 × 12288 × 2 bytes × 36 layers ≈ 10.8 GB.
RMSNorm weights are 0.006% of the model. They are never the memory bottleneck.


## Epilogue Fusion and Sync Barriers: What Actually Happens

Your intuition is on the right track. Let me make it precise.

---

### The simple case: GEMM epilogue WITHOUT RMSNorm

After the tensor core loop finishes, each thread has its output in registers (`rC`):
```python
cute.gemm(tiled_mma, rA, rB, rC)   # tensor cores done — rC holds the result

# Each thread independently writes its portion to global memory
cute.copy(rC, mOut[row_tile, col_tile])
```
No synchronization. Thread 5's output goes to a different address from thread 6's.
They write in parallel with zero coordination.

---

### The fused case: GEMM epilogue WITH RMSNorm

Here is the problem: RMSNorm needs the RMS of the **entire row** before it can normalize even one element. The entire row is spread across all 128 threads. Thread 0 alone doesn't have enough information.

So the fused epilogue adds a reduction step **between the compute and the write**:

```
Phase 1 — compute   (each thread independently)
  rC holds the GEMM output for this thread's elements

Phase 2 — partial sums to SMEM   (each thread writes one value)
  ss = sum(rC[e]^2 for each element e this thread owns)
  smem[tid] = ss

Phase 3 — BARRIER: __syncthreads()
  ← EVERY thread must pause here until ALL 128 threads have written to SMEM
  ← Without this barrier: thread 0 might read SMEM before thread 127 has written

Phase 4 — tree reduce in SMEM   (active threads halve each round)
  for half in [64, 32, 16, 8, 4, 2, 1]:
      if tid < half: smem[tid] += smem[tid + half]
      __syncthreads()    ← barrier after each round (7 barriers total)

  After 7 rounds: smem[0] holds the global sum-of-squares for the full row

Phase 5 — read the result   (all threads)
  inv_rms = rsqrt(smem[0] / D + eps)
  gamma_i  = mW[col_index]

Phase 6 — normalize in registers   (each thread independently)
  for each element e this thread owns:
      rC[e] = rC[e] * inv_rms * gamma_i[e]

Phase 7 — write to global memory   (each thread independently)
  cute.copy(rC_normalized, mOut[row_tile, col_tile])
  ← No barrier needed: each thread writes to different memory addresses
```

---

### Answering your specific question

> "wait based on a memory barrier — synchronization function so that multiple computed results
> will load from shared memory to registers then to global memory all at the same time?"

**The barriers (Phase 3 and the 7 in Phase 4) are mandatory.** They ensure every thread's partial
sum is in SMEM before any thread reads from SMEM. Without them, you'd get race conditions
and wrong answers.

**The "load from SMEM to registers" part** (Phase 5) is just one scalar: `inv_rms`.
Every thread reads the same single value from `smem[0]`. That value is tiny — 4 bytes.

**The "write to global memory"** (Phase 7) is NOT synchronized. Threads write independently.
Within a warp, 32 threads issue 32 stores in the same clock cycle — that's the closest thing
to "at the same time," but it's a hardware parallelism, not a software synchronization.

---

### Why fusing saves memory bandwidth

Without fusion:
```
GEMM kernel:       writes Y = X @ W.T  →  HBM  (one write of [M, N])
RMSNorm kernel:    reads Y ← HBM, writes Y_norm → HBM  (one read + one write)
Total: 3 HBM passes of [M, N]
```

With fusion:
```
Fused kernel:      computes Y in registers, applies RMSNorm in registers,
                   writes Y_norm → HBM  (one write only)
Total: 1 HBM pass of [M, N]
```

At prefill M=2048 for one layer's pre-attention norm:
`2048 × 4096 × 2 bytes = 16 MB` saved per residual+norm pair.
Two per layer × 36 layers = **2.3 GB less HBM traffic per prefill call**.

## RMSNorm Kernels in the Wild

---

### This project (qwen3mma / quack)

Currently only `01_rmsnorm_v0_pytorch.py` exists — the PyTorch baseline.
The CuTeDSL kernels (`sm89_v1_block_per_row`, `sm89_v2_smem_tree_reduce`) are referenced
in the notebooks but not yet written. They are the next step on the optimization ladder.

---

### FlashInfer

FlashInfer is the most commonly referenced library for this kind of fused kernel.
It has two relevant functions:

**`flashinfer.norm.rmsnorm(input, weight, eps)`**
- Standalone RMSNorm
- CUDA C++ kernel, not CuTeDSL
- 256 threads per block, vectorized float4 loads
- Warp shuffle + SMEM hybrid reduction

**`flashinfer.norm.fused_add_rmsnorm(input, residual, weight, eps)`**
- Fuses the residual add with the norm in one kernel
- This is the more important one for transformers

Here is why the fused version matters. In `baseline/block.py`:
```python
# Two separate Python calls today:
x = x + attn_out          # residual add — writes updated x to HBM
x = x + self.mlp(self.ln2(x))  # ln2 reads x from HBM again
```

With FlashInfer's fused kernel:
```python
# One kernel call does both:
x, ln2_input = fused_add_rmsnorm(x, attn_out, ln2.weight, eps)
# x = x + attn_out  (updated in place)
# ln2_input = rmsnorm(x)  (computed in the same pass)
```
Saves one full HBM read of the hidden state. At prefill T=2048: 16 MB saved per layer call.

Source: `flashinfer/csrc/norm.cuh` on GitHub (`flashinfer-ai/flashinfer`).

---

### vLLM

`csrc/layernorm_kernels.cu` — same fused residual+RMSNorm pattern.
Also supports in-place operation where the normalized output overwrites the input buffer.
Not CuTeDSL. Very similar to FlashInfer's implementation.

---

### TensorRT-LLM (TRT-LLM)

TRT-LLM fuses RMSNorm into its attention plugins rather than exposing a standalone kernel.
The normalization is part of a larger fused attention block that processes:
pre-norm → QKV projection → attention → post-norm in one compiled TensorRT engine.
This means the fusion goes even further than just GEMM+RMSNorm — the whole attention sublayer
is one compiled unit.

---

### Liger-Kernel

A Triton-based implementation (`liger-kernel/src/liger_kernel/ops/rms_norm.py`).
Also supports fused residual add. Triton compiles to PTX at runtime, so it's portable
across GPU architectures without writing CUDA C++. Typically 10–20% slower than the
hand-tuned CUDA C++ versions on NVIDIA hardware, but much easier to modify.

---

### Pattern shared across all implementations

Every production RMSNorm kernel does the same four things:

1. **Vectorized loads** — read 4+ BF16 values per instruction (float4 or uint4)
2. **FP32 accumulation** — compute sum-of-squares in FP32 even if input is BF16
3. **Warp shuffle + SMEM hybrid reduce** — shuffles within each warp, SMEM for inter-warp
4. **Fused residual add** — do `x = x + residual` and `norm = rmsnorm(x)` in one pass

The key optimization gap in this project: the CuTeDSL `sm89_v1` and `sm89_v2` kernels
still need to be written to match what FlashInfer and vLLM already do.

# From Token Embedding to RMSNorm: the full picture

## 1. What “input activations at index $i$” means

Think of one token’s hidden state as a vector of numbers:

$$x = [x_0, x_1, x_2, \dots, x_{d-1}]$$

Each $x_i$ is one scalar value in that vector. In Qwen3, the main hidden state is typically length 4096, so this vector has 4096 entries.

Example:

$$x = [3.0, 4.0, 0.0]$$

Then:

- $x_0 = 3.0$
- $x_1 = 4.0$
- $x_2 = 0.0$

RMSNorm does not normalize each entry independently. It first computes one RMS value for the whole vector, then uses that single value to normalize every element:

$$y_i = \frac{x_i}{\mathrm{RMS}(x)} \cdot \gamma_i$$

where

$$\mathrm{RMS}(x) = \sqrt{\frac{1}{d}\sum_{j=0}^{d-1} x_j^2 + \varepsilon}$$

### Intuition

The vector has a certain “energy” or magnitude. RMSNorm rescales that energy to a standard level, then applies a learned per-dimension gain.

---

## 2. What happens after the user query is tokenized and embedded

The full flow is:

1. Text is tokenized into token IDs.
2. Each token ID is looked up in the embedding table.
3. That lookup produces a hidden-state vector.
4. That vector enters the first transformer layer.
5. Inside the layer, RMSNorm is applied before attention and before the MLP path.
6. The hidden state is updated across many layers.
7. A final RMSNorm is applied, then the LM head produces logits for the next token.

### Simple diagram

```text
User text
  |
  v
Token IDs
  |
  v
Embedding lookup  [151936 x 4096]
  |
  v
Hidden state x    [1, 1, 4096]
  |
  v
Layer 0: RMSNorm -> QKV -> Attention -> Residual
  |
  v
Layer 1 ... Layer 35
  |
  v
Final RMSNorm
  |
  v
LM head -> logits -> next token
```

### Bottom line

Yes: the embedding becomes the first hidden-state vector, and that vector is then passed through RMSNorm as part of the transformer stack.

---

## 3. Where the data lives in memory

On an RTX 4080, the important memory levels are:

```text
HBM / VRAM (global memory)
  ├─ model weights
  ├─ embedding table
  ├─ activations / outputs
  └─ KV cache
        |
        v
L2 cache
        |
        v
Shared memory (SMEM)
        |
        v
Registers
```

### Practical interpretation

- The embedding table lives in HBM / VRAM.
- When a token is processed, its embedding vector is loaded from HBM.
- During the kernel run, values are moved into registers and possibly shared memory.
- The normalized output is written back to HBM as a new activation tensor.

So the model weights and activations are mostly in global memory, while computation uses faster on-chip memory temporarily.

---

## 4. Where RMSNorm fits in the computation

For one token and one layer, the flow is:

```text
input activation x  [4096]
  |
  v
RMSNorm
  |
  v
normalized x'       [4096]
  |
  v
projection / attention / MLP
```

The normalization step itself is:

```text
sum of squares -> RMS -> inv_rms -> x_i * inv_rms * gamma_i
```

This means the kernel:

1. reads the activation vector from memory,
2. computes the RMS from the whole vector,
3. applies the normalization,
4. writes the result back.

---

## 5. PyTorch vs Triton vs CUTLASS / CuTeDSL

### PyTorch

- Highest-level interface.
- Easiest to write and inspect.
- Under the hood, PyTorch launches optimized CUDA kernels.
- Often creates extra temporary tensors, which increases HBM traffic.

### Triton

- Lower-level than PyTorch.
- Gives more control over the GPU kernel structure.
- Useful for custom ops and better memory-access patterns.

### CUTLASS / CuTeDSL

- Lowest-level among these options.
- Lets you explicitly manage tiling, shared memory, warp reductions, and tensor-core usage.
- Best when you want to fuse operations like GEMM + RMSNorm into one kernel.

### Why fusion matters

If the workflow is:

```text
GEMM -> write intermediate to HBM -> RMSNorm -> write output to HBM
```

then the GPU does extra memory traffic.

If you fuse the two steps:

```text
GEMM -> RMSNorm in registers -> write final output once
```

then you save bandwidth. On the RTX 4080, that is often the dominant performance win.

---

## 6. Why tensor layout matters

Tensor layout means how values are arranged in memory. Two tensors can contain the same numbers but perform very differently depending on layout.

### Example layout

A hidden-state tensor for one token is often shaped like:

```text
[seq_len, hidden_dim]
```

A good layout keeps nearby values contiguous so the GPU can read them efficiently.

If the layout is poor, the kernel may issue many scattered loads instead of fast contiguous vectorized loads.

### Why this matters on the RTX 4080

- contiguous loads are faster,
- vectorized loads can fetch multiple values at once,
- shared-memory reductions are more efficient when access is regular,
- tensor cores prefer the expected tile structure.

So layout is not a cosmetic detail; it affects how well the GPU can use memory bandwidth and compute throughput.

---

## 7. One-sentence takeaway

In Qwen3 inference, the embedding creates the first hidden vector, RMSNorm is applied repeatedly inside the transformer stack, and performance depends heavily on memory movement, kernel fusion, and tensor layout rather than on raw arithmetic alone.

## Community Optimization Resources for RMSNorm on SM89

### Speedup reference (community benchmarks on SM89-adjacent hardware)

| Optimization | Hardware | Before | After | Speedup | Implementation | What was fused |
|---|---|---|---|---|---|---|
| Triton standalone RMSNorm | RTX 5090 | 4 HBM passes | 1 HBM pass | **5.51×** | Triton | None — reduces passes |
| Fused residual + RMSNorm | RTX 5090 | 2 kernels | 1 kernel | **4.49×** | Triton | Residual add + norm |
| `fused_add_rmsnorm` (FlashInfer) | RTX 4090 | residual + norm separate | 1 kernel | ~3–4× | CUDA C++ | Residual add + norm |
| QK-norm warp shuffle | RTX 3090 | SMEM tree (7 barriers) | warp shuffle (0 SMEM) | ~1.3× | CUDA C++ | Per-head reduction |

**Source:** unsloth/unsloth Triton kernel benchmarks (RMSNorm section); flashinfer-ai/flashinfer `csrc/norm.cuh`

---

### CuTeDSL vs Triton: warp shuffle specificity for SM89

For layer-norm (D=4096), the CuTeDSL kernel can do a **hybrid reduce** that Triton `tl.sum` does not generate:

```
CuTeDSL explicit 5-round shuffle + 4-write SMEM inter-warp:
  Round 1–5: __shfl_xor_sync per warp (0 SMEM writes)
  1 __syncthreads() to write 4 warp-level partials into SMEM
  1 __syncthreads() to broadcast inv_rms
  Total: 2 barriers, 4 SMEM writes, 4 SMEM reads

Triton tl.sum (may generate):
  Full SMEM tree reduce: 128 writes + 7 × (128/2 + ...) reads
  Total: up to 7 barriers, 128 SMEM writes
```

For QK-norm (D=128, one warp fits the entire row):
```python
ss += cute.shfl_xor(ss, 16)
ss += cute.shfl_xor(ss, 8)
ss += cute.shfl_xor(ss, 4)
ss += cute.shfl_xor(ss, 2)
ss += cute.shfl_xor(ss, 1)
inv_rms = cute.rsqrt(cute.shfl_broadcast(ss, 0) / D + eps)
```
**0 bytes of SMEM, 0 `__syncthreads()` calls** — 5 register-to-register passes only.

---

### What to optimize for Qwen3-8B (145 RMSNorm calls per decode step)

| Priority | Optimization | Expected gain | Applies to |
|---|---|---|---|
| 1 | Fuse residual add + RMSNorm | 2–4× per fused call | ln1 and ln2 (72 calls) |
| 2 | Warp-only QK-norm (D=128) | ~1.3× per call | q_norm and k_norm (72 calls) |
| 3 | CUDA graphs (eliminate 145 kernel launches) | 5–20 µs × 145 = 0.7–2.9 ms | All 145 calls at once |
| 4 | Fuse norm into GEMM epilogue | Eliminates 2 HBM passes | ln1+q_proj, ln2+gate_proj |

**FlashInfer reference:** `flashinfer.norm.fused_add_rmsnorm` in `flashinfer-ai/flashinfer`
**Liger-Kernel reference:** `liger_kernel.ops.rms_norm` — Triton-based, portable, ~10% slower than hand-tuned CUDA C++
**vLLM reference:** `vllm/csrc/layernorm_kernels.cu` — same fused residual+norm pattern with in-place support